In [ ]:
import pandas as pd
from tqdm import tqdm
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import time

from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.model_selection import train_test_split

from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.utils import to_categorical
from tensorflow.data import AUTOTUNE
from tensorflow.keras import layers
from tensorflow.keras.applications.inception_v3 import InceptionV3, preprocess_input

from scipy.linalg import sqrtm

import cv2

import warnings
warnings.filterwarnings('ignore')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [6]:
!unzip "/content/drive/MyDrive/processed (1).zip" -d "/content/processed"


Streaming output truncated to the last 5000 lines.
  inflating: /content/processed/processed/train/outside/PsgRFdk2wpeFuJH1HYKc1g.jpg  
  inflating: /content/processed/processed/train/outside/psGrujpCo-p5QzRxyfG6CQ.jpg  
  inflating: /content/processed/processed/train/outside/PSHhyEsMT-Vo311oL-8THg.jpg  
  inflating: /content/processed/processed/train/outside/Psimct8s4tW4Np6BrY2NAw.jpg  
  inflating: /content/processed/processed/train/outside/psIN8b-Rzc-FtpZBczKw_g.jpg  
  inflating: /content/processed/processed/train/outside/pSJaakAwxa0UmiVg8hgL_w.jpg  
  inflating: /content/processed/processed/train/outside/PSjFgnX0IVdGWMUStpUOBg.jpg  
  inflating: /content/processed/processed/train/outside/PsltY9VGKHOpnFjMqAopsw.jpg  
  inflating: /content/processed/processed/train/outside/PSmU_iV5giP-Cb4BnfhqTw.jpg  
  inflating: /content/processed/processed/train/outside/PSNKmotofM0Uygdtc-xTDg.jpg  
  inflating: /content/processed/processed/train/outside/PsnqhaNqpomBi9ElUQU2HQ.jpg  
  inflating: /

In [8]:
import pandas as pd

# Load the preprocessed training metadata
df_train = pd.read_csv('/content/processed/processed/train_meta.csv')

# Preview the first few rows
df_train.head()


,photo_id,business_id,caption,label,image_path
0,_5AXzUhxvmNU0HVS2ECGLg,U_Sw9VC6mCyTbUo2mejS0A,Peppermint cream tea with fruit purée and boba,drink,photos\_5AXzUhxvmNU0HVS2ECGLg.jpg
1,NzBxTwQVB8HTWgM5gyv4wQ,KqF1W-GxTAnOZtcMNS5Wiw,NaN,inside,photos\NzBxTwQVB8HTWgM5gyv4wQ.jpg
2,wk77TsuJbAHCCdPnfWWdRA,RQAF6a0akMiot5lZZnMNNw,NaN,outside,photos\wk77TsuJbAHCCdPnfWWdRA.jpg
3,NpvUCC2ezjtG5yAdccNmjQ,I8lz3u94Y-dWft5fVN3IIA,Beer Flight & Flatbread,food,photos\NpvUCC2ezjtG5yAdccNmjQ.jpg
4,IPCFHtfmWnvB0GQIrvqM-Q,HdTGEuChp8bqRMUjnkqgFA,"Reused paper menus...dried sauce by ""paneer""",menu,photos\IPCFHtfmWnvB0GQIrvqM-Q.jpg


In [10]:
df_train = pd.read_csv('/content/processed/processed/train_meta.csv')  # or correct file name
df_train.head()


,photo_id,business_id,caption,label,image_path
0,_5AXzUhxvmNU0HVS2ECGLg,U_Sw9VC6mCyTbUo2mejS0A,Peppermint cream tea with fruit purée and boba,drink,photos\_5AXzUhxvmNU0HVS2ECGLg.jpg
1,NzBxTwQVB8HTWgM5gyv4wQ,KqF1W-GxTAnOZtcMNS5Wiw,NaN,inside,photos\NzBxTwQVB8HTWgM5gyv4wQ.jpg
2,wk77TsuJbAHCCdPnfWWdRA,RQAF6a0akMiot5lZZnMNNw,NaN,outside,photos\wk77TsuJbAHCCdPnfWWdRA.jpg
3,NpvUCC2ezjtG5yAdccNmjQ,I8lz3u94Y-dWft5fVN3IIA,Beer Flight & Flatbread,food,photos\NpvUCC2ezjtG5yAdccNmjQ.jpg
4,IPCFHtfmWnvB0GQIrvqM-Q,HdTGEuChp8bqRMUjnkqgFA,"Reused paper menus...dried sauce by ""paneer""",menu,photos\IPCFHtfmWnvB0GQIrvqM-Q.jpg


In [11]:
# Load the training metadata
df_train=pd.read_csv('/content/processed/processed/train_meta.csv')

In [12]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 160080 entries, 0 to 160079
Data columns (total 5 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   photo_id     160080 non-null  object
 1   business_id  160080 non-null  object
 2   caption      77525 non-null   object
 3   label        160080 non-null  object
 4   image_path   160080 non-null  object
dtypes: object(5)
memory usage: 6.1+ MB


In [14]:
# Load the training metadata
df_test=pd.read_csv('/content/processed/processed/train_meta.csv')

In [15]:
df_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 160080 entries, 0 to 160079
Data columns (total 5 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   photo_id     160080 non-null  object
 1   business_id  160080 non-null  object
 2   caption      77525 non-null   object
 3   label        160080 non-null  object
 4   image_path   160080 non-null  object
dtypes: object(5)
memory usage: 6.1+ MB


In [16]:
df_train.head(5)

,photo_id,business_id,caption,label,image_path
0,_5AXzUhxvmNU0HVS2ECGLg,U_Sw9VC6mCyTbUo2mejS0A,Peppermint cream tea with fruit purée and boba,drink,photos\_5AXzUhxvmNU0HVS2ECGLg.jpg
1,NzBxTwQVB8HTWgM5gyv4wQ,KqF1W-GxTAnOZtcMNS5Wiw,NaN,inside,photos\NzBxTwQVB8HTWgM5gyv4wQ.jpg
2,wk77TsuJbAHCCdPnfWWdRA,RQAF6a0akMiot5lZZnMNNw,NaN,outside,photos\wk77TsuJbAHCCdPnfWWdRA.jpg
3,NpvUCC2ezjtG5yAdccNmjQ,I8lz3u94Y-dWft5fVN3IIA,Beer Flight & Flatbread,food,photos\NpvUCC2ezjtG5yAdccNmjQ.jpg
4,IPCFHtfmWnvB0GQIrvqM-Q,HdTGEuChp8bqRMUjnkqgFA,"Reused paper menus...dried sauce by ""paneer""",menu,photos\IPCFHtfmWnvB0GQIrvqM-Q.jpg


In [17]:
df_test.head(5)

,photo_id,business_id,caption,label,image_path
0,_5AXzUhxvmNU0HVS2ECGLg,U_Sw9VC6mCyTbUo2mejS0A,Peppermint cream tea with fruit purée and boba,drink,photos\_5AXzUhxvmNU0HVS2ECGLg.jpg
1,NzBxTwQVB8HTWgM5gyv4wQ,KqF1W-GxTAnOZtcMNS5Wiw,NaN,inside,photos\NzBxTwQVB8HTWgM5gyv4wQ.jpg
2,wk77TsuJbAHCCdPnfWWdRA,RQAF6a0akMiot5lZZnMNNw,NaN,outside,photos\wk77TsuJbAHCCdPnfWWdRA.jpg
3,NpvUCC2ezjtG5yAdccNmjQ,I8lz3u94Y-dWft5fVN3IIA,Beer Flight & Flatbread,food,photos\NpvUCC2ezjtG5yAdccNmjQ.jpg
4,IPCFHtfmWnvB0GQIrvqM-Q,HdTGEuChp8bqRMUjnkqgFA,"Reused paper menus...dried sauce by ""paneer""",menu,photos\IPCFHtfmWnvB0GQIrvqM-Q.jpg


In [18]:
# === Paths (change as needed) ===
base_dir = "PHOTOSFILES/processed"
train_dir = os.path.join(base_dir, "train")
test_dir = os.path.join(base_dir, "test")
train_meta_path = os.path.join("train_meta.csv")
test_meta_path = os.path.join("test_meta.csv")

In [19]:
import os
import pandas as pd
# === Function to count images ===
def count_images_in_folder(folder_path, image_exts={'.jpg', '.jpeg', '.png', '.bmp', '.gif'}):
    return sum(
        1 for root, _, files in os.walk(folder_path)
        for file in files if os.path.splitext(file)[1].lower() in image_exts
    )
# === Count images ===
num_train_images = count_images_in_folder(train_dir)
num_test_images = count_images_in_folder(test_dir)





In [20]:
# === Print Summary ===
print(f" Train images found: {num_train_images}")
print(f" Test images found: {num_test_images}")



 Train images found: 0
 Test images found: 0


In [21]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical

In [22]:
import os
import numpy as np
import pandas as pd
from PIL import Image
import tensorflow as tf
from tensorflow.keras import layers

# Set constants
IMG_SIZE = 64  # Resize images to 64x64 for efficiency
NOISE_DIM = 100
BATCH_SIZE = 64
EPOCHS = 50
NUM_CLASSES = 4  # Assuming labels: food, drink, inside, outside


In [23]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical

In [24]:
 #Combine label columns
# Initialize label encoder
label_encoder = LabelEncoder()
all_labels = pd.concat([df_train['label'], df_test['label']], ignore_index=True)


In [25]:
from sklearn.preprocessing import LabelEncoder
import pandas as pd

# Initialize LabelEncoder
label_encoder = LabelEncoder()

# Fit on combined labels to cover all possible classes
label_encoder.fit(pd.concat([df_train['label'], df_test['label']]))

# Now you can safely transform
df_train['label_encoded'] = label_encoder.transform(df_train['label'])
df_test['label_encoded'] = label_encoder.transform(df_test['label'])


In [26]:
def load_images_and_labels(df, image_dir, label_encoder, img_size=(64, 64)):
    images, labels = [], []

    for _, row in tqdm(df.iterrows(), total=len(df), desc="📦 Loading images"):
        label = row['label']
        photo_id = row['photo_id']
        img_path = os.path.join(image_dir, label, photo_id + '.jpg')

        if os.path.exists(img_path):
            try:
                img = Image.open(img_path).convert('RGB').resize(img_size)
                img = np.array(img) / 255.0
                images.append(img)
                labels.append(label_encoder.transform([label])[0])
            except:
                continue

    images = np.array(images)
    labels = to_categorical(labels, num_classes=len(label_encoder.classes_))
    return images, labels


In [27]:
from tensorflow.keras.utils import to_categorical
from PIL import Image
import numpy as np
from tqdm import tqdm

def load_images_and_labels(df, image_dir, label_encoder, img_size=(64, 64), limit=10000):
    images, labels = [], []
    count = 0

    for _, row in tqdm(df.iterrows(), total=len(df), desc="📦 Loading images"):
        if count >= limit:
            break

        label = row['label']
        photo_id = row['photo_id']
        img_path = os.path.join(image_dir, label, photo_id + '.jpg')

        if os.path.exists(img_path):
            try:
                img = Image.open(img_path).convert('RGB').resize(img_size)
                img = np.array(img) / 255.0
                images.append(img)
                labels.append(label_encoder.transform([label])[0])
                count += 1
            except:
                continue

    images = np.array(images, dtype=np.float32)
    labels = to_categorical(labels, num_classes=len(label_encoder.classes_))
    return images, labels


In [28]:
 #One-hot encode
from tensorflow.keras.utils import to_categorical

y_train = to_categorical(df_train['label_encoded'], num_classes=len(label_encoder.classes_))
y_test = to_categorical(df_test['label_encoded'], num_classes=len(label_encoder.classes_))

In [29]:
X_train, y_train = load_images_and_labels(df_train, train_dir, label_encoder, limit=10000)
X_test, y_test = load_images_and_labels(df_test, test_dir, label_encoder, limit=10000)

print("Final Train shape:", X_train.shape, y_train.shape)
print(" Final Test shape:", X_test.shape, y_test.shape)


📦 Loading images: 100%|██████████| 160080/160080 [00:12<00:00, 13240.05it/s]

Final Train shape: (0,) (0, 5)
 Final Test shape: (0,) (0, 5)


In [30]:
import tensorflow as tf
from tensorflow.keras import layers
import numpy as np

# Constants
IMG_SIZE = 64
NOISE_DIM = 100
BATCH_SIZE = 64
EPOCHS = 50
NUM_CLASSES = y_train.shape[1]  # From one-hot shape


Generate Model

In [31]:
def build_generator():
    noise_input = layers.Input(shape=(NOISE_DIM,))
    label_input = layers.Input(shape=(NUM_CLASSES,))

    x = layers.Concatenate()([noise_input, label_input])
    x = layers.Dense(8*8*256, use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU()(x)
    x = layers.Reshape((8, 8, 256))(x)

    x = layers.Conv2DTranspose(128, 5, strides=2, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU()(x)

    x = layers.Conv2DTranspose(64, 5, strides=2, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU()(x)

    output = layers.Conv2DTranspose(3, 5, strides=2, padding='same', activation='tanh')(x)
    return tf.keras.Model([noise_input, label_input], output)

generator = build_generator()


Discriminator Model

In [32]:
def build_discriminator():
    image_input = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    label_input = layers.Input(shape=(NUM_CLASSES,))

    label_embedding = layers.Dense(IMG_SIZE * IMG_SIZE * 1)(label_input)
    label_embedding = layers.Reshape((IMG_SIZE, IMG_SIZE, 1))(label_embedding)

    x = layers.Concatenate()([image_input, label_embedding])
    x = layers.Conv2D(64, 5, strides=2, padding='same')(x)
    x = layers.LeakyReLU()(x)
    x = layers.Dropout(0.3)(x)

    x = layers.Conv2D(128, 5, strides=2, padding='same')(x)
    x = layers.LeakyReLU()(x)
    x = layers.Dropout(0.3)(x)

    x = layers.Flatten()(x)
    output = layers.Dense(1)(x)
    return tf.keras.Model([image_input, label_input], output)

discriminator = build_discriminator()


In [33]:
generator.save("cgan_generator.h5")

Losses & Optimizers

In [34]:
cross_entropy = tf.keras.losses.BinaryCrossentropy(from_logits=True)

def generator_loss(fake_output):
    return cross_entropy(tf.ones_like(fake_output), fake_output)

def discriminator_loss(real_output, fake_output):
    real_loss = cross_entropy(tf.ones_like(real_output), real_output)
    fake_loss = cross_entropy(tf.zeros_like(fake_output), fake_output)
    return real_loss + fake_loss

gen_optimizer = tf.keras.optimizers.Adam(1e-4)
disc_optimizer = tf.keras.optimizers.Adam(1e-4)


Training Step

In [35]:
@tf.function
def train_step(images, labels):
    noise = tf.random.normal([BATCH_SIZE, NOISE_DIM])

    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        generated_images = generator([noise, labels], training=True)

        real_output = discriminator([images, labels], training=True)
        fake_output = discriminator([generated_images, labels], training=True)

        gen_loss = generator_loss(fake_output)
        disc_loss = discriminator_loss(real_output, fake_output)

    gradients_gen = gen_tape.gradient(gen_loss, generator.trainable_variables)
    gradients_disc = disc_tape.gradient(disc_loss, discriminator.trainable_variables)

    gen_optimizer.apply_gradients(zip(gradients_gen, generator.trainable_variables))
    disc_optimizer.apply_gradients(zip(gradients_disc, discriminator.trainable_variables))

    return gen_loss, disc_loss


 Training Loop

In [36]:
def train_cgan(X_train, y_train, epochs=50, batch_size=64):
    # Ensure only 10,000 max samples used ---
    max_samples = 10000
    X_train = X_train[:10000]
    y_train = y_train[:10000]

    steps_per_epoch = X_train.shape[0] // batch_size

    for epoch in range(epochs):
        print(f"\nEpoch {epoch + 1}/{epochs}")
        for i in range(steps_per_epoch):
            idx = np.random.randint(0, X_train.shape[0], batch_size)
            img_batch = X_train[idx]
            label_batch = y_train[idx]
            g_loss, d_loss = train_step(img_batch, label_batch)
        print(f" Gen Loss: {g_loss:.4f} | Disc Loss: {d_loss:.4f}")


In [37]:
discriminator.save("cgan_discriminator.h5")

In [38]:
# Use this if you still have label_encoder loaded from earlier
label_names = list(label_encoder.classes_)  # ['drink', 'food', 'inside', 'outside']
print("Available labels:", label_names)


Available labels: ['drink', 'food', 'inside', 'menu', 'outside']


In [39]:
def generate_synthetic_images(n=100):
    noise = tf.random.normal([n, NOISE_DIM])
    random_labels = tf.random.uniform([n], minval=0, maxval=NUM_CLASSES, dtype=tf.int32)
    onehot_labels = tf.keras.utils.to_categorical(random_labels, NUM_CLASSES)
    fake_images = generator([noise, onehot_labels], training=False)
    return fake_images, onehot_labels


In [40]:
# IS and FID calculation
def calculate_is_fid_optimized(real_images, fake_images):
    # Use InceptionV3 to get 2048-d features
    inception_model = InceptionV3(include_top=False, pooling='avg', input_shape=(299,299,3))

    def preprocess_and_extract(images):
        images_resized = tf.image.resize(images, (299,299))
        images_resized = preprocess_input(images_resized*255)
        features = inception_model.predict(images_resized, verbose=0)
        return features.astype(np.float32)

    # Extract features
    real_features = preprocess_and_extract(real_images)
    fake_features = preprocess_and_extract(fake_images)

    # Compute means and covariances
    mu_real = np.mean(real_features, axis=0)
    mu_fake = np.mean(fake_features, axis=0)
    sigma_real = np.cov(real_features, rowvar=False)
    sigma_fake = np.cov(fake_features, rowvar=False)

    # Compute FID
    diff = mu_real - mu_fake
    cov_sqrt, _ = sqrtm(sigma_real @ sigma_fake, disp=False)
    if np.iscomplexobj(cov_sqrt):
        cov_sqrt = cov_sqrt.real
    fid_score = np.sum(diff**2) + np.trace(sigma_real + sigma_fake - 2*cov_sqrt)

    # Compute dummy IS (or plug your own IS function)
    inception_score = np.random.uniform(1.5, 3.5)

    return inception_score, fid_score

In [41]:
pip install scipy


Load Pretrained InceptionV3 for Feature Extractio

In [42]:
from tensorflow.keras.applications.inception_v3 import InceptionV3, preprocess_input
from tensorflow.keras.models import Model
import tensorflow as tf
import numpy as np
from scipy.linalg import sqrtm

# Load model and cut off before softmax
base_model = InceptionV3(include_top=True, weights='imagenet')
model = Model(inputs=base_model.input, outputs=base_model.get_layer('avg_pool').output)


96112376/96112376 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


Preprocessing Function

In [43]:
def preprocess_images(images):
    # Resize to 299x299 and scale to [-1, 1]
    images = tf.image.resize(images, [299, 299])
    return preprocess_input(images * 255.0)  # Expect input in [0, 255]


Generate Fake and Real Image Batches

In [44]:
def get_fake_images(generator, n_samples=10000):
    noise = tf.random.normal([n_samples, NOISE_DIM])
    labels = tf.random.uniform([n_samples], minval=0, maxval=NUM_CLASSES, dtype=tf.int32)
    onehot_labels = tf.keras.utils.to_categorical(labels, NUM_CLASSES)
    fake_images = generator([noise, onehot_labels], training=False)
    return (fake_images + 1) / 2.0  # Scale from [-1,1] to [0,1]

def get_real_images(X_real, n_samples=100):
    idx = np.random.choice(len(X_real), size=n_samples, replace=False)
    return X_real[idx]


Compute Inception Score (IS)

In [45]:
def inception_score(images, splits=10):
    images = preprocess_images(images)
    preds = base_model.predict(images)
    scores = []

    split_size = len(images) // splits
    for i in range(splits):
        part = preds[i * split_size : (i + 1) * split_size]
        py = np.mean(part, axis=0)
        kl = part * (np.log(part + 1e-10) - np.log(py + 1e-10))
        scores.append(np.exp(np.mean(np.sum(kl, axis=1))))
    return np.mean(scores), np.std(scores)


Compute Fréchet Inception Distance (FID)

In [46]:
def calculate_fid(real_images, fake_images):
    act1 = model.predict(preprocess_images(real_images))
    act2 = model.predict(preprocess_images(fake_images))

    mu1, sigma1 = act1.mean(axis=0), np.cov(act1, rowvar=False)
    mu2, sigma2 = act2.mean(axis=0), np.cov(act2, rowvar=False)

    ssdiff = np.sum((mu1 - mu2)**2.0)
    covmean = sqrtm(sigma1.dot(sigma2))

    if np.iscomplexobj(covmean):
        covmean = covmean.real

    fid = ssdiff + np.trace(sigma1 + sigma2 - 2.0 * covmean)
    return fid


Evalaution

In [48]:
import os
from tensorflow.keras.preprocessing import image
import numpy as np

# Define the test image path
test_dir = '/content/processed/processed/test/'

def load_images_from_folder(folder_path, img_size=(64, 64), limit=None):
    images = []
    count = 0
    for class_folder in os.listdir(folder_path):
        class_path = os.path.join(folder_path, class_folder)
        if not os.path.isdir(class_path):
            continue
        for img_file in os.listdir(class_path):
            img_path = os.path.join(class_path, img_file)
            try:
                img = image.load_img(img_path, target_size=img_size)
                img_array = image.img_to_array(img)
                images.append(img_array)
                count += 1
                if limit and count >= limit:
                    return np.array(images)
            except:
                continue
    return np.array(images)

# Load real test images (e.g., 1000)
X_test = load_images_from_folder(test_dir, limit=1000) / 255.0
print("Loaded images:", X_test.shape)


Loaded images: (1000, 64, 64, 3)


In [49]:
# Sample 3000 images each
fake_images = get_fake_images(generator, n_samples=1000)
real_images = get_real_images(X_test, n_samples=1000)

# Evaluate
is_mean, is_std = inception_score(fake_images)
fid_score = calculate_fid(real_images, fake_images)

print(f" Inception Score: {is_mean:.4f} ± {is_std:.4f}")
print(f" FID Score: {fid_score:.4f}")


32/32 ━━━━━━━━━━━━━━━━━━━━ 263s 8s/step
32/32 ━━━━━━━━━━━━━━━━━━━━ 258s 8s/step
32/32 ━━━━━━━━━━━━━━━━━━━━ 258s 8s/step
 Inception Score: 1.0125 ± 0.0006
 FID Score: 643.0994


In [50]:
def train_cgan(X_train, y_train, epochs=50, batch_size=64):
    X_train = X_train[:10000]
    y_train = y_train[:10000]
    steps_per_epoch = X_train.shape[0] // batch_size

    for epoch in range(epochs):
        print(f"\n Epoch {epoch + 1}/{epochs}")
        for step in range(steps_per_epoch):
            idx = np.random.randint(0, X_train.shape[0], batch_size)
            x_batch = X_train[idx]
            y_batch = y_train[idx]
            g_loss, d_loss = train_step(x_batch, y_batch)

        print(f"Gen Loss: {g_loss:.4f} | Disc Loss: {d_loss:.4f}")
        display_generated_images_per_label(generator, label_encoder, n_per_label=5)


In [51]:
import itertools

# Define your search grid
param_grid = {
    'learning_rate': [0.0002, 0.0001],
    'batch_size': [32, 64],
    'latent_dim': [100, 128],
    'beta_1': [0.5, 0.6]
}

# Generate all combinations of parameters
param_combinations = list(itertools.product(*param_grid.values()))


In [52]:
import numpy as np
import pandas as pd

tuning_results = []

# Loop through each config
for i, (lr, batch_size, latent_dim, beta1) in enumerate(param_combinations):
    print(f"\n Run {i+1}/{len(param_combinations)}")
    print(f"LR={lr}, Batch={batch_size}, Latent={latent_dim}, Beta1={beta1}")

    # Set global variables
    NOISE_DIM = latent_dim

    # Rebuild models
    generator = build_generator()
    discriminator = build_discriminator()

    # Define optimizers
    generator_optimizer = tf.keras.optimizers.Adam(learning_rate=lr, beta_1=beta1)
    discriminator_optimizer = tf.keras.optimizers.Adam(learning_rate=lr, beta_1=beta1)

    # Dummy training function for now (you can replace with real one)
    def dummy_train(generator, discriminator, epochs=2):
        # Simulate score (or replace with loss/FID)
        return np.random.uniform(0.7, 1.0)

    # Train for few epochs and get dummy score
    score = dummy_train(generator, discriminator)

    # Store results
    tuning_results.append({
        'learning_rate': lr,
        'batch_size': batch_size,
        'latent_dim': latent_dim,
        'beta_1': beta1,
        'score': score
    })



 Run 1/16
LR=0.0002, Batch=32, Latent=100, Beta1=0.5

 Run 2/16
LR=0.0002, Batch=32, Latent=100, Beta1=0.6

 Run 3/16
LR=0.0002, Batch=32, Latent=128, Beta1=0.5

 Run 4/16
LR=0.0002, Batch=32, Latent=128, Beta1=0.6

 Run 5/16
LR=0.0002, Batch=64, Latent=100, Beta1=0.5

 Run 6/16
LR=0.0002, Batch=64, Latent=100, Beta1=0.6

 Run 7/16
LR=0.0002, Batch=64, Latent=128, Beta1=0.5

 Run 8/16
LR=0.0002, Batch=64, Latent=128, Beta1=0.6

 Run 9/16
LR=0.0001, Batch=32, Latent=100, Beta1=0.5

 Run 10/16
LR=0.0001, Batch=32, Latent=100, Beta1=0.6

 Run 11/16
LR=0.0001, Batch=32, Latent=128, Beta1=0.5

 Run 12/16
LR=0.0001, Batch=32, Latent=128, Beta1=0.6

 Run 13/16
LR=0.0001, Batch=64, Latent=100, Beta1=0.5

 Run 14/16
LR=0.0001, Batch=64, Latent=100, Beta1=0.6

 Run 15/16
LR=0.0001, Batch=64, Latent=128, Beta1=0.5

 Run 16/16
LR=0.0001, Batch=64, Latent=128, Beta1=0.6


In [53]:
# Convert results to DataFrame
results_df = pd.DataFrame(tuning_results)

# Sort by highest score
results_df = results_df.sort_values(by='score', ascending=False)
display(results_df)


,learning_rate,batch_size,latent_dim,beta_1,score
1,0.0002,32,100,0.6,0.996978
11,0.0001,32,128,0.6,0.991268
15,0.0001,64,128,0.6,0.974154
7,0.0002,64,128,0.6,0.964331
5,0.0002,64,100,0.6,0.945325
13,0.0001,64,100,0.6,0.931316
2,0.0002,32,128,0.5,0.919081
6,0.0002,64,128,0.5,0.914950
12,0.0001,64,100,0.5,0.911839
10,0.0001,32,128,0.5,0.826970
